In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import sqlite3
import pandas as pd

from src.analytics.cashflow_kpis import capital_allocation_pattern

DB_PATH = PROJECT_ROOT / "data" / "nifty100.db"

conn = sqlite3.connect(DB_PATH)

cash = pd.read_sql(
    """
    SELECT
        company_id,
        year,
        operating_activity,
        investing_activity,
        financing_activity
    FROM cashflow
    """,
    conn
)

cash = cash.drop_duplicates(
    subset=["company_id", "year"]
)

cash.head()

,company_id,year,operating_activity,investing_activity,financing_activity
0,TCS,NaN,11615.0,-6038.0,-5729.0
12,ABB,2012.0,101.0,-59.0,-42.0
13,ABB,2014.0,155.0,-144.0,-42.0
14,ABB,2015.0,215.0,-187.0,-58.0
15,ABB,2016.0,249.0,-77.0,-80.0


In [3]:
capital = cash.copy()

capital["cfo_sign"] = capital["operating_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

capital["cfi_sign"] = capital["investing_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

capital["cff_sign"] = capital["financing_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

capital["pattern_label"] = capital.apply(
    lambda x: capital_allocation_pattern(
        x["operating_activity"],
        x["investing_activity"],
        x["financing_activity"]
    ),
    axis=1
)

capital.head()

,company_id,year,operating_activity,investing_activity,financing_activity,cfo_sign,cfi_sign,cff_sign,pattern_label
0,TCS,NaN,11615.0,-6038.0,-5729.0,+,-,-,Reinvestor
12,ABB,2012.0,101.0,-59.0,-42.0,+,-,-,Reinvestor
13,ABB,2014.0,155.0,-144.0,-42.0,+,-,-,Reinvestor
14,ABB,2015.0,215.0,-187.0,-58.0,+,-,-,Reinvestor
15,ABB,2016.0,249.0,-77.0,-80.0,+,-,-,Reinvestor


In [4]:
capital = capital[
    [
        "company_id",
        "year",
        "cfo_sign",
        "cfi_sign",
        "cff_sign",
        "pattern_label"
    ]
]

capital.shape

(1057, 6)

In [7]:
OUTPUT = PROJECT_ROOT / "output"

OUTPUT.mkdir(exist_ok=True)

capital.to_csv(
    OUTPUT / "capital_allocation.csv",
    index=False
)

print("capital_allocation.csv created successfully!")

capital_allocation.csv created successfully!


In [6]:
capital.head(20)

,company_id,year,cfo_sign,cfi_sign,cff_sign,pattern_label
0,TCS,NaN,+,-,-,Reinvestor
12,ABB,2012.0,+,-,-,Reinvestor
13,ABB,2014.0,+,-,-,Reinvestor
14,ABB,2015.0,+,-,-,Reinvestor
15,ABB,2016.0,+,-,-,Reinvestor
16,ABB,2017.0,+,-,-,Reinvestor
17,ABB,2018.0,+,-,-,Reinvestor
18,ABB,2019.0,+,-,-,Reinvestor
19,ABB,2020.0,+,-,-,Reinvestor
20,ABB,2021.0,+,-,-,Reinvestor
